In [ ]:
#파일 위치 확인

import os
print(os.getcwd())
print(os.listdir())

/Users/sangmin/Documents/project/SKN19-Fnal-5team
['jsonl_to_json.ipynb', 'cnslt_cases.jsonl', 'Playwright.py', 'test.py', 'consumer24_cnsl_cases.jsonl', 'test.ipynb']


In [ ]:
# test.ipynb에서 끌어온 url주소를 json파일로 변환

import json
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

INPUT_FILE = "cnslt_cases.jsonl"
OUTPUT_FILE = "cnslt_cases_full.json"

async def main():
    results = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            urls = [json.loads(line)["url"] for line in f]

        for idx, url in enumerate(urls, 1):
            print(f"[{idx}/{len(urls)}] crawling:", url)

            await page.goto(url, wait_until="networkidle")
            html = await page.content()
            soup = BeautifulSoup(html, "html.parser")

            def get_row(label: str) -> str:
                th = soup.find("th", string=label)
                if not th:
                    return ""
                td = th.find_next_sibling("td")
                return td.get_text("\n", strip=True) if td else ""

            data = {
                "url": url,
                "title": get_row("제목"),
                "source": get_row("출처"),
                "category": get_row("분류"),
                "question": get_row("질문"),
                "answer": get_row("답변"),
            }

            results.append(data)

        await browser.close()

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ 저장 완료 → {OUTPUT_FILE}")

# ✅ 노트북에서는 이거!
await main()


[1/75] crawling: https://www.consumer.go.kr/user/ftc/consumer/cnsltcase/114/selectCnsltCaseView.do?prgnCnsltCaseSn=55497&page=1&row=25&searchBgCode=&searchMdCode=&searchSmCode=&searchCnd=&searchWrd=
[2/75] crawling: https://www.consumer.go.kr/user/ftc/consumer/cnsltcase/114/selectCnsltCaseView.do?prgnCnsltCaseSn=50578&page=1&row=25&searchBgCode=&searchMdCode=&searchSmCode=&searchCnd=&searchWrd=
[3/75] crawling: https://www.consumer.go.kr/user/ftc/consumer/cnsltcase/114/selectCnsltCaseView.do?prgnCnsltCaseSn=53321&page=1&row=25&searchBgCode=&searchMdCode=&searchSmCode=&searchCnd=&searchWrd=
[4/75] crawling: https://www.consumer.go.kr/user/ftc/consumer/cnsltcase/114/selectCnsltCaseView.do?prgnCnsltCaseSn=55644&page=1&row=25&searchBgCode=&searchMdCode=&searchSmCode=&searchCnd=&searchWrd=
[5/75] crawling: https://www.consumer.go.kr/user/ftc/consumer/cnsltcase/114/selectCnsltCaseView.do?prgnCnsltCaseSn=45532&page=1&row=25&searchBgCode=&searchMdCode=&searchSmCode=&searchCnd=&searchWrd=
[6/75